# 01. Проверка аддитивной калибровки
Используйте распакованные результаты 00_baseline. Для каждой даты сравнивается фиксированная поправка по первым 14 дням и средняя ошибка сырых прогнозов за предыдущие 7, 14 или 28 дней. Сила скользящей поправки равна 50% или 100%. На ранних датах окно использует всю уже доступную историю weekly. Кандидат выбирается по средней дневной WAPE + |Relative Bias| за 4–17 октября.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
roots = [Path("artifacts/matvey/results-00_baseline"), Path("../../artifacts/matvey/results-00_baseline"), Path("results-00_baseline"), Path("/kaggle/input")]
paths = [p for root in roots if root.is_dir() for p in root.rglob("weekly_predictions.parquet")]
if len(paths) != 1:
    raise ValueError(f"Нужен один распакованный baseline_00: {paths}")
weekly = pd.read_parquet(paths[0])
key = ["cutoff","route_id","horizon"]
if weekly.duplicated(key).any() or not weekly.groupby("cutoff").size().eq(8000).all():
    raise ValueError("Неполная панель weekly")
if not (weekly.model_info_through < weekly.cutoff).all():
    raise ValueError("Нарушена временная граница модели")
err = (weekly.y-weekly.raw_prediction).groupby(weekly.cutoff).mean().sort_index()
dates = err.index
if not dates.equals(pd.date_range(dates[0], periods=42, freq="D")):
    raise ValueError("Ожидались шесть последовательных недель")
known = weekly.groupby("cutoff").timestamp.max().sort_index()
if not (known.iloc[:-1].to_numpy() < dates[1:].to_numpy()).all():
    raise ValueError("Метка предыдущего дня ещё недоступна")
fixed = err.loc[weekly.loc[weekly.period.eq("warmup"), "cutoff"].unique()].mean()
offsets = {"fixed": pd.Series(fixed, index=dates)}
for n in (7,14,28):
    for strength in (0.5,1.0):
        offsets[f"{n}d_x{strength}"] = strength*err.shift(1).rolling(n,min_periods=1).mean()

def day_metrics(frame, name):
    pred = np.maximum(0, frame.raw_prediction + frame.cutoff.map(offsets[name]))
    error = pred - frame.y
    sums = pd.DataFrame(dict(day=frame.cutoff, y=frame.y, absolute=error.abs(), signed=error)).groupby("day").sum()
    return pd.DataFrame(dict(WAPE=sums.absolute/sums.y, bias=sums.signed/sums.y,
                             score=(sums.absolute+sums.signed.abs())/sums.y))

tune = weekly.loc[weekly.period.eq("tuning")]
holdout = weekly.loc[weekly.period.eq("holdout")]
ranking = pd.Series({k:day_metrics(tune,k).score.mean() for k in offsets}).sort_values(kind="stable")
selected = ranking.index[0]
display(ranking.to_frame("tuning_score"))
rows = []
for period, frame in [("tuning",tune),("test",holdout)]:
    for name in dict.fromkeys(["fixed",selected]):
        days = day_metrics(frame,name)
        for group, d in [("все даты",days),("первая неделя",days.iloc[:7]),("вторая неделя",days.iloc[7:]),("субботы",days.loc[days.index.dayofweek==5])]:
            rows.append(dict(period=period, group=group, candidate=name, daily_score=d.score.mean(),
                             daily_WAPE=d.WAPE.mean(), daily_abs_bias=d.bias.abs().mean()))
report = pd.DataFrame(rows)
display(report)
report.to_csv("calibration_report.csv",index=False)
print("Локальная оценка test основана на прогнозах до финального refit. Даты ранее просматривались.")
import json
settings = dict(run_id=weekly.run_id.unique().item(),
                selected=selected, tuning_score=float(ranking.iloc[0]))
Path("calibration_settings.json").write_text(json.dumps(settings, indent=2))